# Part 3: Fabric IQ로 구조화된 데이터 추가하기

Zava DIY 팀은 모든 제품 재고를 Microsoft Fabric으로 관리하지만, 아직 아무도 이를 AI 지식 시스템에 연결하지 않았습니다. 여름 세일이 3주 앞으로 다가온 가운데, 그 문제를 해결하는 것은 바로 여러분입니다. 구매 담당자는 또한 긴급 재입고 예산을 어떻게 에스컬레이션해야 하는지도 알아야 합니다.

**🎯 미션**
- Zava DIY 제품 데이터에 연결된 **Fabric IQ 온톨로지 지식 소스(Fabric IQ Ontology knowledge source)** 추가하기
- 3개 소스로 구성된 지식 베이스(HR 문서 + 건강 문서 + Fabric IQ) 구축하기
- 하나의 하위 답변은 구조화된 제품 재고 데이터에서, 다른 하나는 HR 정책에서 나오는 질문하기


## 빌딩 블록

Part 1과 2에서 지식 베이스는 텍스트 문서를 질의했습니다. 이제 새로운 소스 유형을 추가합니다. Microsoft Fabric의 구조화된 운영 데이터에 연결하는 **Fabric 온톨로지 지식 소스(Fabric Ontology Knowledge Source)** 입니다.

- **Fabric 온톨로지 지식 소스(Fabric Ontology Knowledge Source)**: Fabric 워크스페이스 온톨로지에 연결합니다. 지식 베이스가 런타임에 `Product` 같은 엔터티 유형에 대해 구조화된 질의를 실행합니다.
- **위임 토큰(Delegated token)**: 이 노트북은 `DeviceCodeCredential`을 사용해 현재 사용자를 로그인시키고 Azure AI Search에 대한 위임 토큰을 획득합니다. 검색은 Fabric 온톨로지 같은 보호된 소스를 질의할 때 그 위임된 사용자 ID를 사용합니다.

## 워크플로우

<img src="../img/workflow-part3.png" width="700"/>


## 1단계: 환경 변수 설정

Azure AI Search, Azure OpenAI, Fabric 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 이 파트에 필요한 값이 들어 있습니다.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## 2단계: 로그인 및 위임 토큰 획득

지식 베이스 검색 호출이 질의 소스 액세스에 여러분의 ID를 사용하려면, Fabric 워크스페이스에 액세스 권한이 있는 로그인된 사용자의 OAuth 토큰이 필요합니다.

아래 셀을 실행하여 안내 사이드바의 자격 증명으로 Azure에 로그인하세요. 터미널에 표시되는 URL과 코드를 브라우저에서 입력해 로그인합니다(Codespaces 등 원격 환경에서도 동작). 성공하면 Azure AI Search 범위에 대한 위임 토큰을 획득하고 검색 중 `query_source_authorization`으로 사용하도록 저장합니다.

In [ ]:
from azure.identity import DeviceCodeCredential

user_credential = DeviceCodeCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## 3단계: 세 개의 지식 소스 만들기

이 지식 베이스를 위해 세 개의 지식 소스를 만듭니다. 앞에서 사용한 것과 동일한 두 개의 검색 인덱스 소스에 더해, Fabric 온톨로지 지식 소스 하나입니다.

* `hrdocs-knowledge-source`: `hrdocs` 검색 인덱스를 가리킴
* `healthdocs-knowledge-source`: `healthdocs` 검색 인덱스를 가리킴
* `fabric-ontology-knowledge-source`: 구조화된 운영 데이터에 사용되는 Fabric 워크스페이스와 온톨로지를 가리킴

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR 문서"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava 건강 복리후생 문서"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology 지식 소스",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")


## 4단계: 멀티 소스 + Fabric 지식 베이스 만들기

지식 베이스는 지식 소스를 LLM 및 검색 동작 방식에 대한 지침과 결합합니다.

이 노트북에서 지식 베이스는 연결된 모델이 질문에 직접 답할 수 있도록 `answerSynthesis` `outputMode`를 사용합니다. 또한 모델이 질의 계획과 소스 선택을 도울 수 있도록 `low` `retrievalReasoningEffort`를 사용합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-fabric-ontology-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="색인된 회사 문서와 Zava 구조화된 제품 데이터를 결합한 멀티 소스 지식 베이스",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="구조화된 운영 데이터에는 Fabric Ontology를 사용하고, HR 및 건강 정책 문서에는 검색 인덱스를 사용하세요.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## 5단계: 지식 베이스에 질의

구조화된 Fabric 온톨로지 데이터와 연결된 채팅 모델의 추론을 결합하는 질문을 합니다.

지식 베이스는 에이전트형 검색을 사용해 언제 검색 인덱스를 질의하고 언제 Fabric 온톨로지 소스를 질의할지 결정합니다.

⏱️ Fabric IQ 지식 소스의 증가된 지연 시간 때문에 검색을 완료하는 데 약 40초가 걸립니다.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("저는 여름 세일을 준비하는 Zava 구매 담당자입니다. "
            "Professional Claw Hammer 16oz(SKU: HTHM001600) 제품의 현재 재고 수량은 얼마인가요? "
            "재고 전략과 예산 감독을 담당하는 Zava 직무는 무엇인가요?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
            reranker_threshold=2.0
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


### 활동 로그 검토

이 활동 로그에서는 Fabric IQ에 대한 호출에 해당하는 "fabricOntology" 단계와, 각 검색 인덱스로 보낸 질의에 해당하는 "searchIndex" 단계를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### 참조 검토

Fabric IQ 지식 소스의 경우 참조는 `type: "fabricOntology"`를 가집니다. `sourceData`에는 합성된 답변이 담긴 `fabricAnswer`와 구조화된 데이터 결과가 담긴 `fabricRawData`가 포함됩니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 소스 헌트

위에 출력된 참조를 살펴보세요. 다음을 식별할 수 있나요?

1. 어떤 지식 소스가 **제품 재고 수량** 질문에 답했나요?
2. 어떤 지식 소스가 **재고 및 예산 감독을 담당하는 직무** 질문에 답했나요?

## ✅ 미션 완료

**무엇을 만들었나:**

* ✓ **Fabric IQ 온톨로지 지식 소스(Fabric IQ Ontology Knowledge Source)**: 구조화된 Fabric 제품 데이터를 지식 베이스에 연결하여 지식 베이스가 런타임에 엔터티 질의를 실행할 수 있게 하는 소스.
* ✓ **3개 소스 지식 베이스**: HR 문서, 건강 문서, 라이브 Fabric 운영 데이터에 걸친 단일 지식 베이스로, 에이전트가 각 하위 질문을 적절한 소스로 라우팅함.
* ✓ **구조화/비구조화 하이브리드 검색**: Fabric IQ가 구조화된 제품 엔터티에서 재고 수량 질문에 답했고, HR 문서가 문서 스니펫에서 재고 및 예산 감독 담당 직무 질문에 답했음.

➡️ [Part 4: Work IQ 추가하기](part4-work-iq-to-kb.ipynb)로 계속 진행하세요.